Python is dynamically typed — variables carry no declared type, and errors surface at runtime. Type hints, introduced in Python 3.5 and steadily extended since, let you annotate variables, function parameters, and return values with type information. The interpreter ignores annotations entirely at runtime; they exist for static analysis tools (mypy, pyright, pyrefly), IDEs, and documentation. Used well, type hints catch whole classes of bugs before code runs, make APIs self-documenting, and enable powerful editor support. This notebook covers the full annotation syntax, the `typing` module, generics, protocols, and the practical patterns you will use every day.

## Basic Annotation Syntax

Annotations use a colon after a variable name or parameter and an arrow (`->`) for return types. They are stored in `__annotations__` but are otherwise ignored by Python.

In [ ]:
# Variable annotations
name: str = "Alice"
age: int = 30
height: float = 1.72
active: bool = True

# Annotation without assignment — declares the type but creates no variable
score: int   # valid; 'score' does not exist yet

print(__annotations__)   # {'name': <class 'str'>, 'age': <class 'int'>, ...}

In [ ]:
# Function annotations
def greet(name: str, times: int = 1) -> str:
    return (f"Hello, {name}!\n" * times).rstrip()


print(greet("Bob"))
print(greet("Alice", 3))

# Annotations are stored in __annotations__
print(greet.__annotations__)   # {'name': str, 'times': int, 'return': str}

In [ ]:
# None as a return type — function returns nothing useful
def log(message: str) -> None:
    print(f"[LOG] {message}")


# *args and **kwargs
def total(*values: int, start: int = 0) -> int:
    return start + sum(values)


print(total(1, 2, 3))          # 6
print(total(1, 2, 3, start=10))  # 16

# Each element of *values is int; the parameter itself is a tuple[int, ...]
print(total.__annotations__)

## Built-in Generic Types (Python 3.9+)

From Python 3.9 onward, built-in containers can be parameterised directly in annotations. Before 3.9, you had to import `List`, `Dict`, `Tuple`, etc. from `typing`.

In [ ]:
# Python 3.9+ — use built-in types directly
def first(items: list[int]) -> int | None:
    return items[0] if items else None


def word_count(text: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    for word in text.split():
        counts[word] = counts.get(word, 0) + 1
    return counts


def minmax(values: list[float]) -> tuple[float, float]:
    return min(values), max(values)


print(first([10, 20, 30]))       # 10
print(first([]))                  # None
print(word_count("the cat sat on the mat"))
print(minmax([3.1, 1.4, 2.7]))   # (1.4, 3.1)

In [ ]:
# set, frozenset, type
def unique_words(text: str) -> set[str]:
    return set(text.lower().split())


# Nested generics
def group_by_length(words: list[str]) -> dict[int, list[str]]:
    groups: dict[int, list[str]] = {}
    for w in words:
        groups.setdefault(len(w), []).append(w)
    return groups


print(group_by_length(["cat", "dog", "elephant", "ox", "ant"]))

## `Optional` and the Union Operator

`X | Y` (Python 3.10+) means a value can be either type `X` or type `Y`. `X | None` is the idiomatic way to say "this value might be absent" — previously written as `Optional[X]`.

In [ ]:
from typing import Optional   # still works; X | None is preferred in 3.10+


def find_user(user_id: int) -> dict[str, str] | None:
    db = {1: {"name": "Alice"}, 2: {"name": "Bob"}}
    return db.get(user_id)


# Union of multiple types
def stringify(value: int | float | bool) -> str:
    return str(value)


# Optional is exactly X | None
def get_env(key: str, default: Optional[str] = None) -> Optional[str]:
    import os
    return os.environ.get(key, default)


print(find_user(1))    # {'name': 'Alice'}
print(find_user(99))   # None
print(stringify(3.14)) # 3.14

## `typing` Module Essentials

The `typing` module provides annotations that have no built-in equivalent: `Any`, `Union`, `Literal`, `TypeAlias`, `Final`, `ClassVar`, `TypedDict`, `NamedTuple`, `Callable`, `Sequence`, `Mapping`, and more.

In [ ]:
from typing import Any, Literal, Final, ClassVar


# Any — opts out of type checking for a value
def dump(obj: Any) -> str:
    return repr(obj)


# Literal — only specific values are allowed
def set_log_level(level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]) -> None:
    print(f"Log level set to {level}")


set_log_level("INFO")
# set_log_level("VERBOSE")  # type error: not a valid literal

# Final — a value that must not be reassigned
MAX_RETRIES: Final = 3
PI: Final[float] = 3.141592653589793

# ClassVar — a class-level variable (not an instance variable)
class Registry:
    _count: ClassVar[int] = 0

    def __init__(self, name: str) -> None:
        self.name = name
        Registry._count += 1

    @classmethod
    def count(cls) -> int:
        return cls._count


Registry("a")
Registry("b")
print(Registry.count())    # 2

In [ ]:
from typing import Callable, Sequence, Mapping


# Callable[[arg_types...], return_type]
def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))


print(apply_twice(lambda x: x * 2, 3))   # 12


# Callable[..., ReturnType] — any arguments
def call_and_log(func: Callable[..., str], *args: Any) -> str:
    result = func(*args)
    print(f"Result: {result}")
    return result


# Sequence — any ordered collection (list, tuple, str, range, …)
def total(nums: Sequence[int]) -> int:
    return sum(nums)


print(total([1, 2, 3]))    # works with list
print(total((1, 2, 3)))    # works with tuple


# Mapping — any key-value mapping (dict, defaultdict, …)
def lookup(data: Mapping[str, int], key: str) -> int | None:
    return data.get(key)


print(lookup({"a": 1, "b": 2}, "a"))   # 1

## `TypedDict` and `NamedTuple`

`TypedDict` gives a dict a fixed set of typed keys. `NamedTuple` creates an immutable record with named, typed fields — like a typed version of `collections.namedtuple`.

In [ ]:
from typing import TypedDict, NamedTuple


class Movie(TypedDict):
    title: str
    year: int
    rating: float


def describe(movie: Movie) -> str:
    return f"{movie['title']} ({movie['year']}) — {movie['rating']:.1f}/10"


m: Movie = {"title": "Dune", "year": 2021, "rating": 8.0}
print(describe(m))

# TypedDict with optional keys
class Config(TypedDict, total=False):
    host: str
    port: int
    debug: bool


cfg: Config = {"host": "localhost"}   # port and debug are optional

In [ ]:
class Point(NamedTuple):
    x: float
    y: float
    label: str = ""    # default value


p = Point(1.0, 2.5, label="origin")
print(p)          # Point(x=1.0, y=2.5, label='origin')
print(p.x, p.y)   # field access by name
print(p[0], p[1]) # still works as a tuple

# Immutable — cannot assign to fields
try:
    p.x = 10   # type: ignore
except AttributeError as e:
    print(e)


def distance(a: Point, b: Point) -> float:
    return ((a.x - b.x) ** 2 + (a.y - b.y) ** 2) ** 0.5


print(distance(Point(0, 0), Point(3, 4)))  # 5.0

## Generic Functions with `TypeVar`

A `TypeVar` creates a placeholder type that the checker resolves at each call site. It lets you say "the return type is the same as the input type" without losing specificity by falling back to `Any`.

In [ ]:
from typing import TypeVar

T = TypeVar("T")


def identity(value: T) -> T:
    """Return the value unchanged — preserves the exact type."""
    return value


# The checker knows identity(42) returns int, identity("hi") returns str
x: int = identity(42)
s: str = identity("hello")
print(x, s)


def first(items: list[T]) -> T | None:
    return items[0] if items else None


# Constrained TypeVar — T must be int or float
Number = TypeVar("Number", int, float)


def double(n: Number) -> Number:
    return n * 2


print(double(5))     # 10
print(double(2.5))   # 5.0

In [ ]:
# Bounded TypeVar — T must be a subtype of Comparable
from typing import SupportsLessThan  # hypothetical; use Protocol in practice

# A more realistic bounded TypeVar example
class Comparable:
    pass

C = TypeVar("C", bound="Comparable")


# Generic class
from typing import Generic


class Stack(Generic[T]):
    """A typed stack."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self) -> T | None:
        return self._items[-1] if self._items else None

    def __len__(self) -> int:
        return len(self._items)


s: Stack[int] = Stack()
s.push(1)
s.push(2)
s.push(3)
print(s.pop())    # 3
print(len(s))     # 2
print(s.peek())   # 2

## Protocols — Structural Subtyping

A `Protocol` defines a structural interface: any class that has the required methods and attributes satisfies the protocol, without needing to inherit from it. This is "duck typing made explicit" — the type checker enforces it statically.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...


class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def draw(self) -> str:
        return f"Circle(r={self.radius})"


class Square:
    def __init__(self, side: float) -> None:
        self.side = side

    def draw(self) -> str:
        return f"Square(s={self.side})"


# Neither Circle nor Square inherits from Drawable — they just implement draw()
def render(shape: Drawable) -> None:
    print(shape.draw())


render(Circle(5.0))
render(Square(3.0))

# Runtime check (because of @runtime_checkable)
print(isinstance(Circle(1), Drawable))   # True
print(isinstance("text", Drawable))      # False

In [ ]:
# Protocol with multiple methods and attributes
class Serializable(Protocol):
    def to_dict(self) -> dict[str, Any]: ...

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "Serializable": ...


from typing import Any


class User:
    def __init__(self, name: str, age: int) -> None:
        self.name = name
        self.age  = age

    def to_dict(self) -> dict[str, Any]:
        return {"name": self.name, "age": self.age}

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "User":
        return cls(data["name"], data["age"])


def save(obj: Serializable) -> str:
    import json
    return json.dumps(obj.to_dict())


u = User("Alice", 30)
print(save(u))

## `dataclass` with Type Annotations

`@dataclass` uses field annotations to generate `__init__`, `__repr__`, and `__eq__` automatically. Annotations are not optional here — they drive code generation.

In [ ]:
from dataclasses import dataclass, field
from typing import ClassVar


@dataclass
class Employee:
    name:       str
    department: str
    salary:     float
    skills:     list[str]      = field(default_factory=list)
    _id_counter: ClassVar[int] = 0

    def __post_init__(self) -> None:
        Employee._id_counter += 1
        self.employee_id: int = Employee._id_counter

    def give_raise(self, percent: float) -> None:
        self.salary *= 1 + percent / 100


e1 = Employee("Alice", "Engineering", 90_000.0, ["Python", "SQL"])
e2 = Employee("Bob",   "Design",      75_000.0)

print(e1)
print(e2)

e1.give_raise(10)
print(f"After raise: {e1.salary:,.0f}")

In [ ]:
# frozen=True — immutable dataclass (like NamedTuple but with inheritance support)
@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

    def distance_to(self, other: "Coordinate") -> float:
        """Rough equirectangular distance in degrees."""
        return ((self.lat - other.lat)**2 + (self.lon - other.lon)**2) ** 0.5


a = Coordinate(51.5, -0.1)    # London
b = Coordinate(48.9, 2.3)     # Paris
print(f"Distance: {a.distance_to(b):.2f} deg")

# Frozen dataclasses are hashable — can be used as dict keys / set members
locations = {a, b}
print(len(locations))

## Special Annotation Forms

A few annotation forms control type-checker behaviour in specific ways.

In [ ]:
from typing import cast, TYPE_CHECKING, overload, NoReturn


# cast — tell the type checker to treat a value as a specific type
# Runtime: returns the value unchanged (no conversion happens)
def parse_id(raw: str) -> int:
    return cast(int, int(raw))   # int() already returns int; cast is for the checker


# TYPE_CHECKING — imports only seen by the type checker, not at runtime
# Breaks circular import cycles and avoids expensive imports at startup
if TYPE_CHECKING:
    from pathlib import Path   # only imported when the type checker runs


# NoReturn — a function that never returns normally (always raises or loops forever)
def fail(msg: str) -> NoReturn:
    raise RuntimeError(msg)


# @overload — declare multiple call signatures for one function
# The checker uses the overload stubs; the implementation handles all cases
@overload
def process(x: int) -> int: ...
@overload
def process(x: str) -> str: ...

def process(x: int | str) -> int | str:
    if isinstance(x, int):
        return x * 2
    return x.upper()


print(process(5))       # 10 — checker knows this returns int
print(process("hi"))    # HI — checker knows this returns str

## `Self` and `ParamSpec` (Python 3.11+)

`Self` annotates methods that return an instance of their own class — including subclasses. `ParamSpec` captures the parameter specification of a callable, enabling precise typing of decorators and higher-order functions.

In [ ]:
from typing import Self, ParamSpec, Concatenate
import functools


class Builder:
    def __init__(self) -> None:
        self._parts: list[str] = []

    def add(self, part: str) -> Self:   # returns the same type, even in subclasses
        self._parts.append(part)
        return self

    def build(self) -> str:
        return " ".join(self._parts)


class UpperBuilder(Builder):
    def add(self, part: str) -> Self:
        return super().add(part.upper())


result = UpperBuilder().add("hello").add("world").build()
print(result)   # HELLO WORLD


# ParamSpec preserves the parameter signature through a decorator
P = ParamSpec("P")


def with_logging(func: Callable[P, T]) -> Callable[P, T]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper


@with_logging
def add(a: int, b: int) -> int:
    return a + b


print(add(3, 4))   # the checker knows add still takes (int, int) -> int

## Annotating at Runtime

Annotations are accessible at runtime through `__annotations__`. The `get_type_hints()` function resolves forward references and string annotations to their actual type objects.

In [ ]:
import typing


class Order:
    order_id:  int
    item:      str
    quantity:  int
    unit_price: float

    def __init__(self, order_id: int, item: str,
                 quantity: int, unit_price: float) -> None:
        self.order_id   = order_id
        self.item       = item
        self.quantity   = quantity
        self.unit_price = unit_price

    def total(self) -> float:
        return self.quantity * self.unit_price


# Raw __annotations__ — may contain strings (forward references)
print(Order.__annotations__)

# get_type_hints resolves forward references
hints = typing.get_type_hints(Order)
print(hints)

# Practical use: auto-build a validator from annotations
def validate_fields(obj: object) -> list[str]:
    errors = []
    hints = typing.get_type_hints(type(obj))
    for field_name, expected_type in hints.items():
        value = getattr(obj, field_name, None)
        if value is not None and not isinstance(value, expected_type):
            errors.append(
                f"{field_name}: expected {expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
    return errors


o = Order(1, "widget", 5, 9.99)
print(validate_fields(o))   # []
print(o.total())

## Gradual Typing and Practical Strategy

Type hints do not have to be all-or-nothing. Python supports **gradual typing**: annotate the parts that benefit most first and expand coverage over time.

In [ ]:
# Practical annotation priorities:

# 1. Public API functions — the highest value for documentation and safety
def parse_config(path: str) -> dict[str, str]:
    """Read a .env style key=value file into a dict."""
    result: dict[str, str] = {}
    try:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, _, v = line.partition("=")
                    result[k.strip()] = v.strip()
    except FileNotFoundError:
        pass
    return result


# 2. Use type: ignore sparingly — only when you are certain the checker is wrong
data: list[int] = [1, 2, 3]
mixed = data + ["a"]  # type: ignore[operator]  — intentional

# 3. Use # type: ignore[code] rather than bare # type: ignore
# — the [code] keeps it narrow and searchable

# 4. Avoid over-annotating internals — focus on function boundaries
def process(records: list[dict[str, Any]]) -> list[str]:
    return [r["name"] for r in records if "name" in r]  # internal impl: unannotated


# 5. Use TYPE_CHECKING for heavy or circular imports
from typing import TYPE_CHECKING, Any
if TYPE_CHECKING:
    import numpy as np   # only imported by the checker, not at runtime

print("Strategy demonstrated")

## Summary

- **Annotations** use `:` for variables and parameters, `->` for return types. They are stored in `__annotations__` and ignored by the runtime — tools like mypy and pyright use them.
- **Built-in generics** (Python 3.9+): `list[int]`, `dict[str, float]`, `tuple[int, ...]`, `set[str]` — use built-in types directly, no imports needed.
- **Union** with `|` (Python 3.10+): `int | str`, `str | None`. `Optional[X]` is `X | None`.
- **`typing` module**: `Any` (opt out), `Literal` (specific values), `Final` (no reassignment), `ClassVar` (class-level field), `Callable[[args], ret]`, `Sequence`, `Mapping`.
- **`TypedDict`** gives a dict typed keys. `total=False` makes all keys optional.
- **`NamedTuple`** is an immutable record with named, typed fields and a default value option.
- **`TypeVar`** defines a type placeholder that the checker resolves per call. Constrain with specific types or `bound=` for a supertype bound. `Generic[T]` makes a class generic.
- **Protocols** define structural interfaces. Any class with the required methods satisfies the protocol without inheriting. `@runtime_checkable` enables `isinstance` checks.
- **`@dataclass`** uses annotations to generate `__init__`, `__repr__`, `__eq__`. `frozen=True` adds immutability and hashability.
- **Special forms**: `cast` (tell the checker a type without conversion), `TYPE_CHECKING` (checker-only imports), `NoReturn` (function never returns normally), `@overload` (multiple call signatures).
- **`Self`** annotates methods returning their own class. **`ParamSpec`** preserves parameter signatures through decorators.
- **Gradual typing**: annotate public APIs first, use `# type: ignore[code]` sparingly, and expand coverage incrementally.